# FPL Points Prediction Pipeline

This notebook orchestrates the full two-stage prediction pipeline:
1. **Stage 1**: Predict minutes played for each player
2. **Stage 2**: Use predicted minutes as a feature to predict fantasy points

The pipeline automatically fetches the latest player data and generates predictions for upcoming gameweeks.

## 1. Import Required Libraries

Import all necessary modules for data processing, prediction, and API calls.

In [11]:
# Setup path to allow imports from parent directory
import sys
from pathlib import Path
import os

# Get the parent directory (project root)
notebook_dir = Path.cwd()
if notebook_dir.name == 'models':
    parent_dir = notebook_dir.parent
    os.chdir(parent_dir)  # Change working directory to project root
else:
    parent_dir = notebook_dir

# Add parent directory to Python path
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

print(f"✅ Added {parent_dir} to Python path")
print(f"Current working directory: {Path.cwd()}")

✅ Added /Users/ola/Documents/FPL_Price_Predictor to Python path
Current working directory: /Users/ola/Documents/FPL_Price_Predictor


In [12]:
from utils.get_last_completed_round import (
    get_last_completed_round,
    get_last_completed_round_local,
)
from fpl_api.fetch_updated_player_data import fetch_all_players_data
from utils.get_prediction_data import get_rows_to_predict
from models.minutes.combined_minutes import minutes_prediction_pipeline
import pandas as pd
import joblib
import numpy as np
from utils.team_id_name_map import team_id_name_map

## 2. Check for New Data

Compare the last completed gameweek from the API with local data to determine if we need to fetch updated player statistics.

In [13]:
# Get last completed gameweek from FPL API
try:
    last_completed_round_api = get_last_completed_round()
    print(f"Last completed round (API): {last_completed_round_api}")
except Exception as e:
    print(f"❌ Error getting API data: {e}")
    raise

# Get last completed gameweek from local data
try:
    last_completed_round_local = get_last_completed_round_local()
    print(f"Last completed round (Local): {last_completed_round_local}")
except Exception as e:
    print(f"❌ Error getting local data: {e}")
    last_completed_round_local = 0  # Default fallback

# Fetch new player data if the local data is outdated
if last_completed_round_api > last_completed_round_local:
    print("🔄 Local data is outdated. Fetching new player data from FPL API...")
    fetch_all_players_data()
    print("✅ Player data updated successfully!")
else:
    print("✅ Local data is up to date!")

Last completed round (API): 7
File found with shape: (28106, 45)
Last completed round (Local): 7
✅ Local data is up to date!
File found with shape: (28106, 45)
Last completed round (Local): 7
✅ Local data is up to date!


## 3. Prepare Data for Prediction

Load and prepare the player data with all necessary features for the prediction models.

In [14]:
# Get the rows to predict for upcoming gameweeks
rows_to_predict = get_rows_to_predict(last_completed_round_api)

print(f"Data shape: {rows_to_predict.shape}")
print(f"Gameweeks to predict: {sorted(rows_to_predict['round'].unique())}")

# Save to CSV for inspection
rows_to_predict.to_csv("data/prediction_data/rows_to_predict.csv", index=False)
print("✅ Prediction data saved to data/prediction_data/rows_to_predict.csv")

# Preview the data
rows_to_predict.head()

File found with shape: (28106, 45)
Shape after adding new columns: (28106, 73)
Data shape: (23054, 29)
Gameweeks to predict: [np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38)]
✅ Prediction data saved to data/prediction_data/rows_to_predict.csv
Shape after adding new columns: (28106, 73)
Data shape: (23054, 29)
Gameweeks to predict: [np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int6

,opponent_team,round,minutes,red_cards,ict_index,value,name,team,ewma_points,ewma_minutes,...,pos_MID,rolling_avg_minutes,next_is_home,next_fixture_attack_rating,next_fixture_defense_rating,self_team_attack_rating,self_team_defense_rating,next_fixture_atk_def_ratio,next_fixture_def_atk_ratio,minutes_next
4330,7,7,0,0.0,0.0,55.0,A.Becker,Liverpool,1.8,54.0,...,False,60.0,1,1110.0,1140.0,1290,1380,1.13,1.24,0.0
4331,3,7,0,0.0,0.0,39.0,A.García,Aston Villa,0.0,0.0,...,False,0.0,0,1100.0,1160.0,1200,1300,1.03,1.18,0.0
4332,10,7,29,0.0,0.7,45.0,A.Jimenez,Bournemouth,0.4,36.0,...,False,48.7,0,1120.0,1160.0,1160,1200,1.00,1.07,0.0
4333,16,7,0,0.0,0.0,39.0,A.Murphy,Newcastle,0.0,0.0,...,False,0.0,0,1090.0,1210.0,1170,1320,0.97,1.21,0.0
4334,2,7,0,0.0,0.0,44.0,A.Ramsey,Burnley,0.0,0.0,...,True,0.0,1,1050.0,1100.0,1050,1050,0.95,1.00,0.0


## 4. Stage 1: Predict Minutes Played

Use the minutes prediction model (classifier + regressor) to predict how many minutes each player will play.

In [15]:
# Run the minutes prediction pipeline
predicted_minutes = minutes_prediction_pipeline(
    rows_to_predict=rows_to_predict, 
    last_completed_round=last_completed_round_api
)

print(f"Minutes predictions generated for {len(predicted_minutes)} player-gameweek combinations")
print(f"\nMinutes prediction stats:")
print(f"  Average: {predicted_minutes['predicted_minutes'].mean():.1f}")
print(f"  Min: {predicted_minutes['predicted_minutes'].min():.0f}")
print(f"  Max: {predicted_minutes['predicted_minutes'].max():.0f}")

# Preview minutes predictions
predicted_minutes.head(10)

File found with shape: (28106, 45)
Max classifier prediction: 1.0
Min classifier prediction: 0.0
Max regressor prediction: 90.0
Min regressor prediction: 35.635479411996464
Minutes predictions generated for 743 player-gameweek combinations

Minutes prediction stats:
  Average: 31.4
  Min: 0
  Max: 90


,name,team,position,value,status,predicted_minutes,raw_classifier_proba,raw_regressor_minutes
4330,A.Becker,Liverpool,GK,55.0,unavailable,0,0.674554,84.992458
4331,A.García,Aston Villa,DEF,39.0,available,0,0.000000,76.293372
4332,A.Jimenez,Bournemouth,DEF,45.0,available,56,0.767296,72.862926
4333,A.Murphy,Newcastle,DEF,39.0,available,0,0.000000,75.435508
4334,A.Ramsey,Burnley,MID,44.0,unavailable,0,0.000000,70.038273
4335,Aaronson,Leeds,MID,54.0,available,72,0.914718,78.535402
4336,Abbott,Nott'm Forest,DEF,39.0,available,4,0.056312,74.178202
4337,Abdullahi,Sunderland,FWD,44.0,available,0,0.000000,69.638347
4338,Acheampong,Chelsea,DEF,40.0,available,43,0.566369,75.052612
4339,Adama,Fulham,MID,53.0,available,32,0.655962,48.954107


## 5. Merge Predicted Minutes with Features

Combine the predicted minutes with the original features to prepare for points prediction.

In [16]:
# Ensure no duplicate 'minutes_next' column before merging
rows_to_predict.drop(columns=["minutes_next"], inplace=True, errors="ignore")

# Merge predicted minutes back into the original rows_to_predict DataFrame
rows_with_mins = pd.merge(
    rows_to_predict,
    predicted_minutes[["name", "team", "position", "predicted_minutes"]],
    on=["name", "team"],
    how="left",
)

# Rename predicted_minutes to minutes_next for the points model
rows_with_mins.rename(columns={"predicted_minutes": "minutes_next"}, inplace=True)

print(f"Merged data shape: {rows_with_mins.shape}")
print(f"✅ Successfully merged predicted minutes with player features")

# Save identifiers for later use
output_columns = [
    "name",
    "team",
    "position",
    "value",
    "round",
    "minutes_next",
    "opponent_team",
]
identifiers_df = rows_with_mins[output_columns].copy()

identifiers_df.head()

Merged data shape: (23054, 30)
✅ Successfully merged predicted minutes with player features


,name,team,position,value,round,minutes_next,opponent_team
0,A.Becker,Liverpool,GK,55.0,7,0,7
1,A.García,Aston Villa,DEF,39.0,7,0,3
2,A.Jimenez,Bournemouth,DEF,45.0,7,56,10
3,A.Murphy,Newcastle,DEF,39.0,7,0,16
4,A.Ramsey,Burnley,MID,44.0,7,0,2


## 6. Stage 2: Predict Fantasy Points

Use the stacking ensemble model to predict fantasy points using all features including predicted minutes.

In [17]:
# Remove identifier columns to prepare for prediction
features_for_prediction = rows_with_mins.drop(
    columns=["name", "team", "opponent_team"], errors="ignore"
)

# Load the points prediction model and feature order
try:
    points_model = joblib.load("data/saved_models/stacking_model.pkl")
    points_feature_order = joblib.load("data/saved_models/feature_order.pkl")
    print("✅ Successfully loaded stacking model and feature order")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Please ensure the points model and its feature order are saved.")

# Reorder columns to match the training feature order
features_for_prediction = features_for_prediction[points_feature_order]

print(f"Features for prediction shape: {features_for_prediction.shape}")
print(f"Number of features: {len(points_feature_order)}")

✅ Successfully loaded stacking model and feature order
Features for prediction shape: (23054, 26)
Number of features: 26


In [18]:
# Predict using the stacking ensemble model
predicted_points_log = points_model.predict(features_for_prediction)

# Transform back from log scale and ensure non-negative values
predicted_points = np.expm1(predicted_points_log)
predicted_points = predicted_points.round(1).clip(min=0)

print(f"✅ Generated {len(predicted_points)} point predictions")
print(f"\nPoints prediction stats:")
print(f"  Average: {predicted_points.mean():.2f}")
print(f"  Min: {predicted_points.min():.1f}")
print(f"  Max: {predicted_points.max():.1f}")
print(f"  Median: {np.median(predicted_points):.2f}")

✅ Generated 23054 point predictions

Points prediction stats:
  Average: 1.40
  Min: 0.0
  Max: 7.8
  Median: 1.00


## 7. Create Final Predictions DataFrame

Combine all predictions with player identifiers and format for output.

In [19]:
# Create final predictions DataFrame with all relevant information
final_df = identifiers_df[
    ["name", "team", "position", "value", "round", "opponent_team", "minutes_next"]
].copy()

# Add predicted points
final_df["predicted_points"] = predicted_points

# Map opponent_team IDs to team names
final_df["opponent_team"] = final_df["opponent_team"].map(team_id_name_map())

# Increment round by 1 to reflect the upcoming round being predicted
final_df["round"] = final_df["round"] + 1

# For each player, shift opponent_team to show next round's opponent
final_df["opponent_team"] = final_df.groupby("name")["opponent_team"].shift(-1)

# Rename columns for clarity
final_df.rename(columns={
    "opponent_team": "next_opponent",
    "minutes_next": "predicted_minutes"
}, inplace=True)

# Convert value to standard FPL format (divide by 10)
final_df["value"] = final_df["value"] / 10.0

print(f"✅ Final predictions DataFrame created with {len(final_df)} rows")
print(f"Columns: {list(final_df.columns)}")

✅ Final predictions DataFrame created with 23054 rows
Columns: ['name', 'team', 'position', 'value', 'round', 'next_opponent', 'predicted_minutes', 'predicted_points']


## 8. Sort and Display Results

Sort predictions by gameweek and points to show top performers for each upcoming round.

In [20]:
# Sort by round (ascending) then predicted points (descending)
final_df.sort_values(
    by=["round", "predicted_points"], 
    ascending=[True, False], 
    inplace=True
)

# Display top 20 predictions
print("Top 20 Predicted Players:")
final_df.head(20)

Top 20 Predicted Players:


,name,team,position,value,round,next_opponent,predicted_minutes,predicted_points
281,Haaland,Man City,FWD,14.4,8,Everton,86,7.3
415,M.Salah,Liverpool,MID,14.5,8,Man Utd,81,7.1
323,Isak,Liverpool,FWD,10.6,8,Man Utd,68,5.5
226,Foden,Man City,MID,8.1,8,Everton,85,5.2
576,Rice,Arsenal,MID,6.5,8,Fulham,83,5.2
574,Reijnders,Man City,MID,5.8,8,Everton,76,4.9
534,O’Reilly,Man City,DEF,4.9,8,Everton,81,4.8
40,Anthony,Burnley,MID,5.7,8,Leeds,77,4.7
239,Gabriel,Arsenal,DEF,6.2,8,Fulham,90,4.7
240,Gakpo,Liverpool,MID,7.5,8,Man Utd,75,4.7


## 9. Analyze Predictions by Position

Break down predictions by player position to identify top performers in each category.

In [29]:
# View top players by position for the next gameweek
next_gw = final_df['round'].min()
next_gw_data = final_df[final_df['round'] == next_gw]

print(f"\n=== Top Players by Position for Gameweek {next_gw} ===\n")

# Get position columns - they are not one-hot encoded. Group by 'position' column instead.
GKs = next_gw_data[next_gw_data['position'] == 'GK']
DEFs = next_gw_data[next_gw_data['position'] == 'DEF']
MIDs = next_gw_data[next_gw_data['position'] == 'MID']
FWDs = next_gw_data[next_gw_data['position'] == 'FWD']

# Display top 5 players for each position
position_groups = {
    'Goalkeepers': GKs,
    'Defenders': DEFs,
    'Midfielders': MIDs,
    'Forwards': FWDs
}

if position_groups:
    for position_name, position_players in position_groups.items():
        if len(position_players) > 0:
            top_3 = position_players.nlargest(3, 'predicted_points')[['name', 'predicted_points', 'predicted_minutes']]
            print(f"\n{position_name}:")
            print(top_3.to_string(index=False))


=== Top Players by Position for Gameweek 8 ===


Goalkeepers:
      name  predicted_points  predicted_minutes
 Henderson               4.4                 90
      Raya               4.2                 90
Donnarumma               4.0                 90

Defenders:
    name  predicted_points  predicted_minutes
O’Reilly               4.8                 81
 Gabriel               4.7                 90
Gvardiol               4.6                 85

Midfielders:
   name  predicted_points  predicted_minutes
M.Salah               7.1                 81
  Foden               5.2                 85
   Rice               5.2                 83

Forwards:
   name  predicted_points  predicted_minutes
Haaland               7.3                 86
   Isak               5.5                 68
Ekitiké               4.5                 63


## 10. Save Predictions to CSV

Export the predictions to a CSV file for use in the Streamlit app or further analysis.

In [32]:
# Save the final predictions
output_path = "data/prediction_data/predicted_points_with_minutes.csv"
final_df.to_csv(output_path, index=False)

print(f"\n✅ Predictions saved to: {output_path}")
print(f"📊 Total predictions: {len(final_df)}")
print(f"🎯 Gameweeks covered: {sorted(final_df['round'].unique())}")


✅ Predictions saved to: data/prediction_data/predicted_points_with_minutes.csv
📊 Total predictions: 23054
🎯 Gameweeks covered: [np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39)]


## 11. Prediction Summary Statistics

View overall statistics about the predictions to understand the distribution and key insights.

In [35]:
print("\n=== Prediction Summary Statistics ===\n")

# Overall statistics
print("Overall Predicted Points:")
print(final_df['predicted_points'].describe())

print("\n\nOverall Predicted Minutes:")
print(final_df['predicted_minutes'].describe())

# Count of players predicted to play (>0 minutes) per gameweek
print("\n\nPlayers Expected to Play by Gameweek:")
for gw in sorted(final_df['round'].unique()):
    gw_data = final_df[final_df['round'] == gw]
    playing = len(gw_data[gw_data['predicted_minutes'] > 0])
    total = len(gw_data)
    avg_points = gw_data['predicted_points'].mean()
    print(f"  GW {gw}: {playing}/{total} players ({playing/total*100:.1f}%) - Avg Points: {avg_points:.2f}")

# High-value predictions
print("\n\nHigh-Value Predictions (>7.5 points):")
high_value = final_df[final_df['predicted_points'] > 7.5]
print(f"  {len(high_value)} predictions across {high_value['name'].nunique()} unique players")


=== Prediction Summary Statistics ===

Overall Predicted Points:
count    23054.000000
mean         1.399623
std          1.582028
min          0.000000
25%          0.000000
50%          1.000000
75%          2.800000
max          7.800000
Name: predicted_points, dtype: float64


Overall Predicted Minutes:
count    23054.000000
mean        31.356554
std         34.474293
min          0.000000
25%          0.000000
50%         16.000000
75%         65.000000
max         90.000000
Name: predicted_minutes, dtype: float64


Players Expected to Play by Gameweek:
  GW 8: 428/743 players (57.6%) - Avg Points: 1.40
  GW 9: 428/743 players (57.6%) - Avg Points: 1.38
  GW 10: 428/743 players (57.6%) - Avg Points: 1.39
  GW 11: 428/743 players (57.6%) - Avg Points: 1.40
  GW 12: 428/743 players (57.6%) - Avg Points: 1.40
  GW 13: 428/743 players (57.6%) - Avg Points: 1.40
  GW 14: 428/743 players (57.6%) - Avg Points: 1.40
  GW 15: 428/743 players (57.6%) - Avg Points: 1.40
  GW 16: 428/743 pla